# Stance detection op TweetEval

Alle 5 stance topics, 3 representaties x 4 modellen, 5-fold CV met 3 seeds, macro-F1 + Bayesiaanse vergelijking.
Code staat in `src/`.

## Setup

In [ ]:
import sys
import warnings
from collections import Counter, OrderedDict
from pathlib import Path

warnings.simplefilter(action="ignore", category=FutureWarning)

# notebook staat in notebooks/, de repo-root is één map hoger
root = Path.cwd().parent
sys.path.insert(0, str(root))

import part1
from src.data import LABEL_NAMES, load_stance_all, make_cv_splits
from src.evaluate import (
    TOKENIZER_MIN_FREQUENCY, TOKENIZER_VOCAB_SIZE,
    compare_systems, majority_baseline_f1, run_cross_validation, summary_table,
)
from src.plots import plot_bayes_comparisons, plot_score_heatmap
from src.features import ENCODER_FACTORIES, BagOfWords, TfidfEncoder, TokenIndicator
from src.models import MODEL_RUNNERS

## Data

Train split van alle topics. `#SemST` eruit (staat in elke tweet), topic als tag ervoor omdat against/favor per topic iets anders betekent.

In [ ]:
stance_dataset = load_stance_all()
train_texts = stance_dataset["train"]["text"]
print(f"{len(train_texts)} train-tweets; de eerste:\n{train_texts[0]}")
Counter(stance_dataset["train"]["topic"])

## BPE stap voor stap

Pretokenisatie

In [ ]:
prepared_corpus = part1.bpe.prepare_corpus(train_texts)
prepared_corpus[:5]

Karakter vocab + UNK

In [ ]:
char_vocab = {ch for token in prepared_corpus for ch in token}
vocab = char_vocab | {"\u2047"}  # '⁇' voor karakters die we later nog nooit gezien hebben
sorted(vocab)

Pretokens tellen

In [ ]:
word_counts = Counter(tuple(word) for word in prepared_corpus)
word_counts.most_common(10)

Paren tellen, meest frequente paar

In [ ]:
pair_counts = Counter()
for symbols, frequency in word_counts.items():
    for left, right in zip(symbols, symbols[1:]):
        pair_counts[(left, right)] += frequency

# bij gelijke telling: lexicografische volgorde als tie-breaker
best_pair = min(pair_counts, key=lambda pair: (-pair_counts[pair], pair))
best_pair, pair_counts[best_pair]

Eerste merge

In [ ]:
merges = OrderedDict()
merges[best_pair] = pair_counts[best_pair]
merges

## Tokenizer

HF BPE op alle train tweets, alleen om te kijken. In de experimenten wordt hij per fold getraind.

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, TOKENIZER_VOCAB_SIZE, TOKENIZER_MIN_FREQUENCY)
tokenizer.encode(train_texts[0]).tokens

## Representaties

Indicator (0/1), bag-of-words (counts), TF-IDF.

In [ ]:
encoders_demo = {
    "indicator": TokenIndicator(tokenizer),
    "bow": BagOfWords(tokenizer),
    "tfidf": TfidfEncoder(tokenizer, train_texts),
}
n_tokens = len(tokenizer.encode(train_texts[0]).ids)
print(f"tweet 0 heeft {n_tokens} tokens")
for name, encoder in encoders_demo.items():
    vector = encoder(train_texts[0])
    print(f"{name:10s} dim={len(vector)}  som={vector.sum():.2f}  niet-nul={int((vector != 0).sum())}")

## Labels

Against is ~48%, daarom macro-F1 ipv accuracy.

In [ ]:
Counter(stance_dataset["train"]["label"])  # 0=none, 1=against, 2=favor

## Experimenten

Modellen:
- `logistic`
- `mlp`: 1 hidden layer (128)
- `mlp_regularized`: hidden 32, dropout 0.5, weight decay 1e-3, batch 16
- `stance_mlp`: LeakyReLU + input dropout (20% tokens weg)

Alle MLP's trainen met dezelfde functie. Tokenizer, TF-IDF en model worden per fold alleen op de train data gefit.

In [ ]:
cv_runs = make_cv_splits(stance_dataset)
f1_scores = run_cross_validation(cv_runs, ENCODER_FACTORIES, MODEL_RUNNERS, num_classes=len(LABEL_NAMES))

## Resultaten

Gemiddelde macro-F1 over 15 runs (5 folds x 3 seeds).

In [ ]:
encoder_names = list(ENCODER_FACTORIES)
model_names = list(MODEL_RUNNERS)
results_table = summary_table(f1_scores, encoder_names, model_names)
baseline = majority_baseline_f1(cv_runs)

plot_score_heatmap(f1_scores, encoder_names, model_names, baseline);

Architecturen vergelijken (met beste representatie). ROPE = 0.01, conclusie vanaf 95%.

TODO: ROPE motiveren

In [ ]:
best_encoder = results_table.max(axis=0).idxmax()
architectures = [f"{model}_{best_encoder}" for model in model_names]
comparison = compare_systems(f1_scores, architectures, runs=len(cv_runs))
plot_bayes_comparisons(comparison, f"Architecturen (allemaal met {best_encoder})", strip=f"_{best_encoder}");

Representaties vergelijken (met beste model)

In [ ]:
best_model = results_table.max(axis=1).idxmax()
representations = [f"{best_model}_{enc}" for enc in encoder_names]
comparison = compare_systems(f1_scores, representations, runs=len(cv_runs))
plot_bayes_comparisons(comparison, f"Representaties (allemaal met {best_model})", strip=f"{best_model}_");